# 03 — Model Comparison Figures (Ch 4.3, 4.4)

Produces:
- **Table 4.2** — All 10 models Dense metrics (LaTeX)
- **Fig 4.5** v1 bar / v2 horizontal / v3 grouped multi-metric — Dense NDCG@10 across models
- **Fig 4.6** v1 / v2 labelled — model size vs Δ NDCG@10 scatter
- **Table 4.3** — BM25 best configs per model (LaTeX)
- **Fig 4.7** v1 line / v2 line+points / v3 heatmap — repetition sweep
- **Fig 4.8** v1 scatter / v2 slope — Dense gain vs BM25 gain

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from _helpers import *

In [ ]:
dense = pd.read_csv(DATA_RAW / 'model_comparison_dense.csv')
bm25  = pd.read_csv(DATA_RAW / 'model_comparison_bm25.csv')
rep   = pd.read_csv(DATA_RAW / 'exp11_ndcg10.csv')
DENSE_BASELINE = float(dense.loc[dense.model == 'mDPR (no QE)', 'ndcg_10'].iloc[0])
BM25_BASELINE  = float(bm25.loc[bm25.model == 'BM25 baseline (no QE)', 'n1_ndcg10'].iloc[0])
print(f'Dense baseline = {DENSE_BASELINE},  BM25 baseline = {BM25_BASELINE}')

## Table 4.2 — Dense models

In [ ]:
t = dense.copy()
t['delta'] = t['ndcg_10'] - DENSE_BASELINE
t = t.sort_values('ndcg_10', ascending=False)
cols = ['model','params_B','ndcg_10','recall_10','recall_100','mrr','delta','note']
(OUTPUT_PDF / 'table_4_2.tex').write_text(
    t[cols].to_latex(index=False, float_format='%.4f', na_rep='-', column_format='lccccccl'),
    encoding='utf-8')
print('saved: table_4_2.tex')
t[cols]

## Fig 4.5 — Dense NDCG@10 across models

In [ ]:
# Exclude baseline + dropped, sort
active = dense[~dense.model.isin(['mDPR (no QE)', 'ALLaM-7B'])].copy()
active = active.sort_values('ndcg_10', ascending=False).reset_index(drop=True)

# v1 — vertical sorted bar
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(active.model, active.ndcg_10, color=[color_for_model(m) for m in active.model], edgecolor='black')
ax.axhline(DENSE_BASELINE, color='#1f6f8a', linestyle='--', linewidth=1, label=f'mDPR baseline ({DENSE_BASELINE:.3f})')
ax.set_ylabel('NDCG@10')
ax.set_ylim(0, max(active.ndcg_10) * 1.1)
ax.legend(loc='upper right')
plt.xticks(rotation=30, ha='right')
save_fig(fig, 'fig_4_5_models_bar_v1')

# v2 — horizontal sorted bar (often better for long model names)
fig, ax = plt.subplots(figsize=(6, 5))
active_h = active.sort_values('ndcg_10')
ax.barh(active_h.model, active_h.ndcg_10, color=[color_for_model(m) for m in active_h.model], edgecolor='black')
ax.axvline(DENSE_BASELINE, color='#1f6f8a', linestyle='--', linewidth=1, label=f'mDPR baseline')
ax.set_xlabel('NDCG@10')
ax.legend(loc='lower right')
save_fig(fig, 'fig_4_5_models_bar_v2_h')

# v3 — grouped bar across metrics
fig, ax = plt.subplots(figsize=(8, 4))
models = active.model.tolist()
x = np.arange(len(models))
w = 0.27
metrics = [('NDCG@10', 'ndcg_10'), ('Recall@10', 'recall_10'), ('MRR', 'mrr')]
for i, (label, col) in enumerate(metrics):
    ax.bar(x + (i-1)*w, active[col].fillna(0), w, label=label,
           color=['#1f6f8a','#8c8c8c','#b3b3b3'][i], edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(models, rotation=30, ha='right')
ax.set_ylabel('Score')
ax.legend(loc='upper right', ncol=3)
save_fig(fig, 'fig_4_5_models_grouped_v3')

## Fig 4.6 — Model size vs Δ NDCG@10

In [ ]:
size_df = dense[~dense.model.isin(['mDPR (no QE)', 'ALLaM-7B', 'Aya 8B CSQE'])].copy()
size_df['delta'] = size_df.ndcg_10 - DENSE_BASELINE
size_df = size_df.dropna(subset=['params_B'])

# v1 — plain scatter
fig, ax = plt.subplots()
ax.scatter(size_df.params_B, size_df.delta, s=60, color='#1f6f8a')
ax.set_xlabel('Model size (billion parameters)')
ax.set_ylabel('NDCG@10 gain over mDPR baseline')
ax.axhline(0, color='#8c8c8c', linewidth=0.7)
save_fig(fig, 'fig_4_6_size_v1')

# v2 — labelled with trendline
fig, ax = plt.subplots()
ax.scatter(size_df.params_B, size_df.delta, s=60, color='#1f6f8a', zorder=3)
for _, r in size_df.iterrows():
    ax.annotate(r.model, (r.params_B, r.delta), xytext=(4, 4), textcoords='offset points', fontsize=8)
z = np.polyfit(size_df.params_B, size_df.delta, 1)
xs = np.linspace(size_df.params_B.min(), size_df.params_B.max(), 50)
ax.plot(xs, np.polyval(z, xs), '--', color='#8c8c8c', linewidth=1,
         label=f'Linear fit: slope={z[0]:.4f}')
ax.set_xlabel('Model size (billion parameters)')
ax.set_ylabel('NDCG@10 gain over mDPR baseline')
ax.axhline(0, color='#8c8c8c', linewidth=0.7)
ax.legend(loc='lower right')
save_fig(fig, 'fig_4_6_size_v2_labelled')

## Table 4.3 — BM25 best configs

In [ ]:
(OUTPUT_PDF / 'table_4_3.tex').write_text(
    bm25.to_latex(index=False, float_format='%.4f', column_format='lccccc'), encoding='utf-8')
print('saved: table_4_3.tex')
bm25

## Fig 4.7 — BM25 repetition sweep

In [ ]:
# Long-form: model x config (n=1..10 + beta=2,4,6)
n_cols = ['n=1','n=3','n=5','n=7','n=10']
beta_cols = ['\u03b2=2','\u03b2=4','\u03b2=6']
models = [m for m in rep['Model'] if m != 'BM25 baseline (no QE)']

# v1 — n sweep multi-line
fig, ax = plt.subplots(figsize=(7, 4))
x_n = [1, 3, 5, 7, 10]
for i, m in enumerate(models):
    ys = rep.loc[rep.Model == m, n_cols].values.flatten()
    ax.plot(x_n, ys, marker='o', label=m, alpha=0.85, linewidth=1.5, color=color_for_model(m))
ax.axhline(BM25_BASELINE, color='#1f6f8a', linestyle=':', linewidth=1, label='BM25 baseline')
ax.set_xlabel('n (query repetitions)')
ax.set_ylabel('NDCG@10')
ax.legend(loc='lower right', fontsize=7, ncol=2)
save_fig(fig, 'fig_4_7_repetition_v1')

# v2 — n sweep + beta markers
fig, ax = plt.subplots(figsize=(7, 4))
for i, m in enumerate(models):
    ys = rep.loc[rep.Model == m, n_cols].values.flatten()
    ax.plot(x_n, ys, marker='o', label=m, alpha=0.85, linewidth=1.5, color=color_for_model(m))
    bs = rep.loc[rep.Model == m, beta_cols].values.flatten()
    ax.scatter([12.5, 13.5, 14.5], bs, marker='s', s=22, alpha=0.7)
ax.axvline(11, color='#8c8c8c', linewidth=0.5, linestyle=':')
ax.set_xticks([1,3,5,7,10,12.5,13.5,14.5])
ax.set_xticklabels(['n=1','n=3','n=5','n=7','n=10','\u03b2=2','\u03b2=4','\u03b2=6'])
ax.set_ylabel('NDCG@10')
ax.legend(loc='lower right', fontsize=7, ncol=2)
save_fig(fig, 'fig_4_7_repetition_v2')

# v3 — heatmap
all_cols = n_cols + beta_cols
M = rep.loc[rep.Model.isin(models), ['Model'] + all_cols].set_index('Model')
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(M.values, cmap='gray_r', aspect='auto', vmin=0.3, vmax=0.6)
ax.set_xticks(range(len(all_cols)))
ax.set_xticklabels(all_cols)
ax.set_yticks(range(len(M.index)))
ax.set_yticklabels(M.index)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M.values[i, j]
        color = 'white' if v > 0.5 else 'black'
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7, color=color)
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='NDCG@10')
save_fig(fig, 'fig_4_7_repetition_v3_heat')

## Fig 4.8 — Dense gain vs BM25 gain

In [ ]:
# Join dense + bm25 on model. The matching is by approximate name.
import re
def slug(s): return re.sub(r'[^a-z0-9]', '', s.lower())
d = dense[~dense.model.isin(['mDPR (no QE)', 'ALLaM-7B', 'Aya 8B CSQE'])].copy()
d['slug'] = d.model.apply(slug)
b = bm25[bm25.model != 'BM25 baseline (no QE)'].copy()
b['slug'] = b.model.apply(slug)
merged = d.merge(b[['slug','n1_ndcg10','best_ndcg10']], on='slug', how='inner')
merged['dense_delta'] = merged['ndcg_10'] - DENSE_BASELINE
merged['bm25_n1_delta'] = merged['n1_ndcg10'] - BM25_BASELINE
merged['bm25_best_delta'] = merged['best_ndcg10'] - BM25_BASELINE
print(merged[['model','dense_delta','bm25_n1_delta','bm25_best_delta']])

# v1 scatter using current technique (n=1, what the thesis baseline reported)
fig, ax = plt.subplots()
ax.scatter(merged.dense_delta, merged.bm25_n1_delta, s=60, color='#1f6f8a', zorder=3)
for _, r in merged.iterrows():
    ax.annotate(r.model, (r.dense_delta, r.bm25_n1_delta), xytext=(4,4), textcoords='offset points', fontsize=8)
ax.axhline(0, color='#8c8c8c', linewidth=0.7)
ax.axvline(0, color='#8c8c8c', linewidth=0.7)
ax.set_xlabel('Δ NDCG@10 on Dense (Query2Doc vs mDPR baseline)')
ax.set_ylabel('Δ NDCG@10 on BM25 (Q2D n=1 vs BM25 baseline)')
save_fig(fig, 'fig_4_8_gains_v1')

# v2 slope chart: Dense Δ -> BM25 Δ per model
fig, ax = plt.subplots(figsize=(6, 5))
for _, r in merged.iterrows():
    ax.plot([0, 1], [r.dense_delta, r.bm25_n1_delta], '-o', alpha=0.8)
    ax.annotate(r.model, (1.02, r.bm25_n1_delta), fontsize=8, va='center')
ax.axhline(0, color='#8c8c8c', linewidth=0.7)
ax.set_xticks([0, 1]); ax.set_xticklabels(['Dense', 'BM25 (n=1)'])
ax.set_ylabel('NDCG@10 gain over baseline')
ax.set_xlim(-0.1, 1.5)
save_fig(fig, 'fig_4_8_gains_v2_slope')